# Baselines

**Objetivo:** criar uma régua simples. O LightGBM só se justifica se superar estes resultados.

- **Regra por valor:** revisar as compras de maior valor.
- **Regressão logística:** colunas definidas na D9.

**Regra:** treino só com os dados de treino e resultados só na validação. O teste não entra em nenhum cálculo.

In [1]:
from pathlib import Path
import re
import sys

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RAIZ = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
sys.path.insert(0, str(RAIZ / "src"))
from divisao_temporal import calcular_cortes, dividir_por_tempo
from metricas import selecionar_revisadas, valor_capturado

ARQUIVO = RAIZ / "data" / "processed" / "treino.parquet"
nomes = pq.ParquetFile(ARQUIVO).schema_arrow.names
COLUNAS_C = [n for n in nomes if re.fullmatch(r"C\d+", n)]
COLUNAS_ID = [n for n in nomes if re.fullmatch(r"id_\d+", n)]
COLUNAS_CATEGORIA = ["ProductCD", "card4", "card6"]

tabela = pd.read_parquet(
    ARQUIVO,
    columns=["TransactionDT", "isFraud", "TransactionAmt", *COLUNAS_CATEGORIA, *COLUNAS_C, *COLUNAS_ID],
)
tabela["tem_identidade"] = tabela[COLUNAS_ID].notna().any(axis=1).astype("int8")
tabela = tabela.drop(columns=COLUNAS_ID)

corte_treino, corte_validacao = calcular_cortes(tabela["TransactionDT"])
treino, validacao, _ = dividir_por_tempo(tabela, corte_treino, corte_validacao)
del _, tabela  # o teste sai da memória antes de qualquer treino ou medição

print(f"Treino: {len(treino):,} | Validação: {len(validacao):,}")
print(f"Menor valor em C no treino: {treino[COLUNAS_C].min().min()}")

Treino: 354,324 | Validação: 118,108
Menor valor em C no treino: 0.0


## 1. Regressão logística

O valor e as colunas C têm muitos números pequenos e poucos muito grandes. O logaritmo reduz essa diferença, e a padronização deixa as colunas na mesma escala. Campos vazios de cartão viram a categoria "(vazio)", e categorias novas na validação são ignoradas.

In [2]:
def preparar(parte: pd.DataFrame) -> pd.DataFrame:
    x = np.log1p(parte[COLUNAS_C].astype("float64"))
    x["log_valor"] = np.log1p(parte["TransactionAmt"].astype("float64"))
    x["tem_identidade"] = parte["tem_identidade"]
    for coluna in COLUNAS_CATEGORIA:
        x[coluna] = parte[coluna].astype("object").fillna("(vazio)")
    return x


colunas_numericas = [*COLUNAS_C, "log_valor", "tem_identidade"]
modelo = Pipeline(
    [
        (
            "preparo",
            ColumnTransformer(
                [
                    ("numericas", StandardScaler(), colunas_numericas),
                    ("categorias", OneHotEncoder(handle_unknown="ignore"), COLUNAS_CATEGORIA),
                ]
            ),
        ),
        ("logistica", LogisticRegression(max_iter=1000)),
    ]
)
modelo.fit(preparar(treino), treino["isFraud"])
print(f"Iterações usadas: {modelo.named_steps['logistica'].n_iter_[0]} de 1000")

rotulos = validacao["isFraud"].astype("int8")
valores = validacao["TransactionAmt"].astype("float64")
probabilidade = pd.Series(modelo.predict_proba(preparar(validacao))[:, 1], index=validacao.index)

Iterações usadas: 39 de 1000


## 2. Qualidade geral na validação

PR-AUC: quanto maior, melhor. Uma ordenação ao acaso fica perto da taxa de fraude da validação. Brier: quanto menor, melhor, e mede se as probabilidades estão certas. A regra por valor não produz probabilidade, então não tem Brier. A linha de referência usa a taxa de fraude do treino como probabilidade para todas as compras.

In [3]:
taxa_treino = float(treino["isFraud"].mean())
pd.DataFrame(
    {
        "Referência (sem modelo)": {
            "PR-AUC": float(rotulos.mean()),
            "Brier": brier_score_loss(rotulos, np.full(len(rotulos), taxa_treino)),
        },
        "Regra por valor": {"PR-AUC": average_precision_score(rotulos, valores), "Brier": np.nan},
        "Regressão logística": {
            "PR-AUC": average_precision_score(rotulos, probabilidade),
            "Brier": brier_score_loss(rotulos, probabilidade),
        },
    }
).T.round(4)

,PR-AUC,Brier
Referência (sem modelo),0.0390,0.0375
Regra por valor,0.0479,NaN
Regressão logística,0.4357,0.0278


## 3. Resultado por capacidade de revisão

Capacidade é a parte das compras da validação que a equipe consegue revisar. Revisando ao acaso, a captura de valor esperada seria próxima da própria capacidade.

In [4]:
linhas = []
for nome, pontuacao in [("Regra por valor", valores), ("Regressão logística", probabilidade)]:
    for capacidade in [0.005, 0.01, 0.03, 0.05]:
        revisadas = selecionar_revisadas(pontuacao, capacidade)
        linhas.append(
            {
                "Modelo": nome,
                "Capacidade (%)": 100 * capacidade,
                "Revisões": len(revisadas),
                "Precisão (%)": 100 * rotulos.loc[revisadas].mean(),
                "Recall (%)": 100 * rotulos.loc[revisadas].sum() / rotulos.sum(),
                "Valor capturado (%)": valor_capturado(rotulos, pontuacao, valores, capacidade),
            }
        )
pd.DataFrame(linhas).set_index(["Modelo", "Capacidade (%)"]).round(2)

Revisões  Precisão (%)  Recall (%)  \
Modelo              Capacidade (%)                                       
Regra por valor     0.5                  591          3.21        0.41   
                    1.0                 1182          4.23        1.08   
                    3.0                 3544          7.93        6.09   
                    5.0                 5906          8.69       11.13   
Regressão logística 0.5                  591         91.88       11.78   
                    1.0                 1182         86.29       22.12   
                    3.0                 3544         51.33       39.45   
                    5.0                 5906         36.10       46.24   

                                    Valor capturado (%)  
Modelo              Capacidade (%)                       
Regra por valor     0.5                            6.73  
                    1.0                           12.14  
                    3.0                           37.60  
                    5.0                           51.83  
Regressão logística 0.5                            3.84  
                    1.0                           11.39  
                    3.0                           26.65  
                    5.0                           34.48

## 4. Diagnóstico: regressão logística sem as colunas C

As colunas C vêm pré-calculadas pela Vesta, e não dá para auditar se usam informação futura. Este teste mostra quanto do resultado da logística depende delas. Ele não prova se existe vazamento.

In [5]:
colunas_sem_c = ["log_valor", "tem_identidade"]
modelo_sem_c = Pipeline(
    [
        (
            "preparo",
            ColumnTransformer(
                [
                    ("numericas", StandardScaler(), colunas_sem_c),
                    ("categorias", OneHotEncoder(handle_unknown="ignore"), COLUNAS_CATEGORIA),
                ]
            ),
        ),
        ("logistica", LogisticRegression(max_iter=1000)),
    ]
)
modelo_sem_c.fit(preparar(treino), treino["isFraud"])
probabilidade_sem_c = pd.Series(modelo_sem_c.predict_proba(preparar(validacao))[:, 1], index=validacao.index)

display(
    pd.DataFrame(
        {
            nome: {
                "PR-AUC": average_precision_score(rotulos, pontuacao),
                "Brier": brier_score_loss(rotulos, pontuacao),
            }
            for nome, pontuacao in [("Com colunas C", probabilidade), ("Sem colunas C", probabilidade_sem_c)]
        }
    ).T.round(4)
)

linhas_c = []
for nome, pontuacao in [("Com colunas C", probabilidade), ("Sem colunas C", probabilidade_sem_c)]:
    for capacidade in [0.005, 0.01, 0.03, 0.05]:
        revisadas = selecionar_revisadas(pontuacao, capacidade)
        linhas_c.append(
            {
                "Modelo": nome,
                "Capacidade (%)": 100 * capacidade,
                "Precisão (%)": 100 * rotulos.loc[revisadas].mean(),
                "Recall (%)": 100 * rotulos.loc[revisadas].sum() / rotulos.sum(),
                "Valor capturado (%)": valor_capturado(rotulos, pontuacao, valores, capacidade),
            }
        )
pd.DataFrame(linhas_c).set_index(["Modelo", "Capacidade (%)"]).round(2)

,PR-AUC,Brier
Com colunas C,0.4357,0.0278
Sem colunas C,0.1351,0.0356


Precisão (%)  Recall (%)  Valor capturado (%)
Modelo        Capacidade (%)                                               
Com colunas C 0.5                    91.88       11.78                 3.84
              1.0                    86.29       22.12                11.39
              3.0                    51.33       39.45                26.65
              5.0                    36.10       46.24                34.48
Sem colunas C 0.5                    26.57        3.40                 2.69
              1.0                    25.04        6.42                 4.17
              3.0                    20.17       15.51                 6.47
              5.0                    17.80       22.79                 8.48

## Conclusões dos baselines

- A regressão logística ordena fraudes muito melhor que a regra por valor: PR-AUC de 0,4357, contra 0,0479 da regra e 0,0390 de uma ordenação ao acaso. O Brier da logística é 0,0278, contra 0,0375 da referência sem modelo.
- A logística tem a maior precisão em todas as capacidades, de 91,88% em 0,5% a 36,10% em 5%. A regra por valor fica entre 3,21% e 8,69%.
- A regra por valor captura mais dinheiro nas quatro capacidades: 6,73%, 12,14%, 37,60% e 51,83%, contra 3,84%, 11,39%, 26,65% e 34,48% da logística.
- Acertar mais fraudes não significa capturar mais dinheiro. A logística ordena pela chance de fraude e ignora quanto cada compra vale. A regra por valor faz o contrário.
- A hipótese é que combinar chance e valor supere os dois critérios. Isso será testado na Etapa 7, comparando a ordenação por `P(fraude)` com `P(fraude) × valor`.
- Quase todo o desempenho da logística vem das colunas C. Sem elas, a PR-AUC cai de 0,4357 para 0,1351, a precisão em 1% cai de 86,29% para 25,04% e o valor capturado em 3% cai de 26,65% para 6,47%.
- Métricas melhores com as colunas C não mostram se elas são sinal real ou informação futura, porque um vazamento também melhoraria as métricas na validação.
- **Limites:** as colunas C vêm pré-calculadas pela Vesta, e não dá para auditar se usam informação futura, então o desempenho da logística pode estar inflado. Os custos de revisão ainda não entram. Estes resultados valem só para a validação.